In [9]:
import polars as pl

import nwec.utility_reporting.arrearage_counts
import nwec.utility_reporting.arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 12
COLS_PER_MONTH = 1
SHEET_SEARCH_STRING = "disconnections"
NUM_DISCONNECTIONS_SEARCH_STRING = "number of disconnections"
NUM_NOTICES_SEARCH_STRING = "number of customers by customer class receiving disconnection notices"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.AVISTA.code}_{YEAR}_Q{QUARTER}.xlsx"
source_date_format = "%Y-%m-%d %H:%M:%S"

In [10]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
_, start_index = nwec.utils.excel.find_cell_by_string(df, NUM_DISCONNECTIONS_SEARCH_STRING)
disconnections = df.select(df.columns[start_index : start_index + NUM_MONTHS * COLS_PER_MONTH])

# Number of Disconnections


In [ ]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(disconnections, source_date_format)
disconnections = disconnections.tail(-date_row)  # remove rows before the date row
disconnections = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(
    disconnections, source_date_format
)
disconnections = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(df, disconnections)
disconnections = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(disconnections, Utility.AVISTA)


# Number of Disconnection Notices


In [14]:
_, start_index = nwec.utils.excel.find_cell_by_string(df, NUM_NOTICES_SEARCH_STRING)
notices = df.select(df.columns[start_index : start_index + NUM_MONTHS * COLS_PER_MONTH])

In [17]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(notices, source_date_format)
notices = notices.tail(-date_row)  # remove rows before the date row
notices = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(notices, source_date_format)
notices = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(df, notices)
notices = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(notices, Utility.AVISTA)


# Save Results


In [ ]:
# nwec.utility_reporting.arrearage_counts.save_processed_arrearage_counts(arrearage_counts)